load the frozen dataset and the inclusion rules approved on 2026-11-13

In [1]:
import csv
import json
import re
from collections import Counter
from html import escape
from pathlib import Path

import gemmi
from IPython.display import HTML, display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_root = project_root / "data"
raw_root = data_root / "raw" / "alpha_synuclein_fibrils"
manifest = json.loads((data_root / "fibril_manifest.json").read_text())
atlas_entries = {
    entry["pdb_id"]: entry
    for entry in manifest["atlas_entries"]
    if entry["selected"]
}
assert set(atlas_entries) == set(manifest["structure_checks"])
approved_scope = {
    "approved_on": "2026-11-13",
    "approved_by": "user",
    "dataset_version": "092458e487fed389912256a345645526dff1251729f5eb8bba1f21c205a83683",
    "purpose": "metadata curation",
    "species": "Homo sapiens",
    "constructs": "retain variants and genuine truncations with annotations",
    "aggregation_states": ["FILAMENT", "HELICAL ARRAY"],
    "supporting_evidence": "linked manuscript required; unresolved evidence stays pending",
    "species_conflicts": ["9K23", "9K24", "9LON"],
}
assert manifest["dataset_version"] == approved_scope["dataset_version"], "new dataset needs inclusion review"
print(f"source dataset: {manifest['dataset_version']}")
print(f"inclusion rules approved by {approved_scope['approved_by']} on {approved_scope['approved_on']}")

source dataset: 092458e487fed389912256a345645526dff1251729f5eb8bba1f21c205a83683
inclusion rules approved by user on 2026-11-13


read the deposited species, sequence lengths and annotations without inferring missing values

In [2]:
def alpha_entity_values(category, field, alpha_ids):
    return [
        value
        for entity_id, value in zip(
            category.get("entity_id", category.get("id", [])),
            category.get(field, []),
        )
        if entity_id in alpha_ids and value
    ]


review_rows = []
for pdb_id, atlas_entry in sorted(atlas_entries.items()):
    block = gemmi.cif.read_file(str(raw_root / "coordinates" / f"{pdb_id}.cif")).sole_block()
    metadata = json.loads((raw_root / "pdb_metadata" / f"{pdb_id}.json").read_text())
    entities = block.get_mmcif_category("_entity.")
    alpha_ids = {
        entity_id
        for entity_id, description in zip(entities["id"], entities["pdbx_description"])
        if re.search(r"(alpha|α)[ -]?synuclein", description or "", re.IGNORECASE)
    }
    species = set()
    for category_name, field in (
        ("_entity_src_gen.", "pdbx_gene_src_scientific_name"),
        ("_entity_src_nat.", "pdbx_organism_scientific"),
        ("_pdbx_entity_src_syn.", "organism_scientific"),
    ):
        species.update(alpha_entity_values(block.get_mmcif_category(category_name), field, alpha_ids))
    sequences = alpha_entity_values(
        block.get_mmcif_category("_entity_poly."), "pdbx_seq_one_letter_code_can", alpha_ids
    )
    lengths = sorted({len("".join(sequence.split())) for sequence in sequences})
    citation_doi = metadata.get("rcsb_primary_citation", {}).get("pdbx_database_id_DOI")
    reference_urls = [
        f"https://doi.org/{citation_doi}" if citation_doi else "",
        atlas_entry["reference_url"] or "",
    ]
    publication_url = next(
        (url for url in reference_urls if re.match(r"https?://(?:dx\.)?doi\.org/10\.(?!2210/)", url)),
        "",
    )
    review_rows.append({
        "pdb_id": pdb_id,
        "species": " | ".join(sorted(species)) or "unknown",
        "deposited_lengths": " | ".join(map(str, lengths)) or "unknown",
        "pdb_mutations": " | ".join(alpha_entity_values(entities, "pdbx_mutation", alpha_ids)) or "unknown",
        "pdb_fragment": " | ".join(alpha_entity_values(entities, "pdbx_fragment", alpha_ids)) or "unknown",
        "atlas_origins": atlas_entry["origins_text"],
        "ordered_residues": atlas_entry["ordered_residues_text"],
        "aggregation_state": metadata.get("em_experiment", {}).get("aggregation_state") or "unknown",
        "citation_doi": citation_doi or "unknown",
        "publication_url": publication_url,
        "review_flags": " | ".join(manifest["structure_checks"][pdb_id]["review_flags"]),
    })

apply the approved scope and preserve existing decisions and reasons

In [3]:
review_path = data_root / "fibril_inclusion_review.csv"
previous_decisions = {}
if review_path.exists():
    with review_path.open(newline="") as source:
        previous_rows = list(csv.DictReader(source))
    previous_decisions = {row["pdb_id"]: row for row in previous_rows}
    assert len(previous_decisions) == len(previous_rows), "duplicate review identifiers"
    assert set(previous_decisions) <= set(atlas_entries), "review contains entries outside this snapshot"

for row in review_rows:
    previous = previous_decisions.get(row["pdb_id"], {})
    if previous.get("decision", "pending") != "pending" or previous.get("reason", "").strip():
        row["decision"] = previous["decision"]
        row["reason"] = previous["reason"]
    elif row["pdb_id"] in approved_scope["species_conflicts"]:
        row["decision"] = "pending"
        row["reason"] = "species conflict: atlas says human; pdb annotation and deposited sequence indicate mouse"
    elif row["species"] == "unknown":
        row["decision"] = "pending"
        row["reason"] = "protein species needs supporting evidence"
    elif row["species"] != approved_scope["species"]:
        row["decision"] = "exclude"
        row["reason"] = "confirmed mouse alpha-synuclein is outside the approved human scope"
    elif row["aggregation_state"] not in approved_scope["aggregation_states"]:
        row["decision"] = "pending"
        row["reason"] = "aggregation state needs fibril evidence"
    elif any(flag != "fibril state needs review" for flag in row["review_flags"].split(" | ") if flag):
        row["decision"] = "pending"
        row["reason"] = "source identity or coordinate validation needs review"
    elif not row["publication_url"]:
        row["decision"] = "pending"
        row["reason"] = "supporting manuscript link unresolved in cached pdb and atlas records"
    else:
        row["decision"] = "include"
        row["reason"] = "human entry with a linked manuscript; eligible for metadata curation"
        if row["aggregation_state"] == "HELICAL ARRAY":
            row["reason"] += "; helical array accepted under the approved fibril rule"
    assert row["decision"] in {"pending", "include", "exclude"}, row["pdb_id"]
    assert row["decision"] == "pending" or row["reason"].strip(), f"reason required: {row['pdb_id']}"

pending_path = review_path.with_suffix(".csv.part")
with pending_path.open("w", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=list(review_rows[0]))
    writer.writeheader()
    writer.writerows(review_rows)
pending_path.replace(review_path)

print(f"approved scope applied to {review_path.relative_to(project_root)}; pending entries require evidence")

approved scope applied to data/fibril_inclusion_review.csv; pending entries require evidence


summarise the entries; deposited lengths describe sequences and unknown annotations do not imply wild type

In [4]:
print(f"entries: {len(review_rows)}")
for field in ("species", "deposited_lengths", "decision"):
    print(f"{field}: {dict(Counter(row[field] for row in review_rows))}")
known_references = {row["publication_url"] for row in review_rows if row["publication_url"]}
print(f"distinct supporting manuscript links: {len(known_references)}")
print(f"entries without a supporting link: {sum(not row['publication_url'] for row in review_rows)}")
print(f"entries with review flags: {sum(bool(row['review_flags']) for row in review_rows)}")

entries: 225
species: {'Homo sapiens': 220, 'Mus musculus': 5}
deposited_lengths: {'67': 1, '140': 179, '141': 14, '98': 1, '136': 8, '63': 3, '121': 1, '55': 3, '100': 1, '65': 3, '90': 1, '134': 1, '99': 2, '96': 1, '62': 2, '5 | 104': 1, '61': 1, '139': 1, '69': 1}
decision: {'include': 170, 'pending': 53, 'exclude': 2}
distinct supporting manuscript links: 60
entries without a supporting link: 50
entries with review flags: 13


show the original flags and the inclusion decisions under the approved rules

In [5]:
def display_fibril_entries(rows, columns):
    header = "".join(f"<th>{escape(column.replace('_', ' '))}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(row[column]))}</td>" for column in columns) + "</tr>"
        for row in rows
    )
    display(HTML(
        f'<div style="max-height: 500px; overflow: auto"><table><thead><tr>{header}</tr></thead>'
        f"<tbody>{body}</tbody></table></div>"
    ))


display_fibril_entries(
    [row for row in review_rows if row["review_flags"]],
    ["pdb_id", "aggregation_state", "review_flags", "publication_url", "decision", "reason"],
)

pdb id,aggregation state,review flags,publication url,decision,reason
8X7B,HELICAL ARRAY,fibril state needs review,https://doi.org/10.1073/pnas.2321633121,include,human entry with a linked manuscript; eligible for metadata curation; helical array accepted under the approved fibril rule
8X7L,HELICAL ARRAY,fibril state needs review,https://doi.org/10.1073/pnas.2321633121,include,human entry with a linked manuscript; eligible for metadata curation; helical array accepted under the approved fibril rule
8X7M,HELICAL ARRAY,fibril state needs review,https://doi.org/10.1073/pnas.2321633121,include,human entry with a linked manuscript; eligible for metadata curation; helical array accepted under the approved fibril rule
8X7O,HELICAL ARRAY,fibril state needs review,https://doi.org/10.1073/pnas.2321633121,include,human entry with a linked manuscript; eligible for metadata curation; helical array accepted under the approved fibril rule
8X7P,HELICAL ARRAY,fibril state needs review,https://doi.org/10.1073/pnas.2321633121,include,human entry with a linked manuscript; eligible for metadata curation; helical array accepted under the approved fibril rule
8X7Q,HELICAL ARRAY,fibril state needs review,https://doi.org/10.1073/pnas.2321633121,include,human entry with a linked manuscript; eligible for metadata curation; helical array accepted under the approved fibril rule
8X7R,HELICAL ARRAY,fibril state needs review,https://doi.org/10.1073/pnas.2321633121,include,human entry with a linked manuscript; eligible for metadata curation; helical array accepted under the approved fibril rule
8ZLI,HELICAL ARRAY,fibril state needs review,https://doi.org/10.1073/pnas.2321633121,include,human entry with a linked manuscript; eligible for metadata curation; helical array accepted under the approved fibril rule
8ZLO,HELICAL ARRAY,fibril state needs review,https://doi.org/10.1073/pnas.2321633121,include,human entry with a linked manuscript; eligible for metadata curation; helical array accepted under the approved fibril rule
8ZLP,HELICAL ARRAY,fibril state needs review,https://doi.org/10.1073/pnas.2321633121,include,human entry with a linked manuscript; eligible for metadata curation; helical array accepted under the approved fibril rule


show all entries and their current inclusion decisions

In [ ]:
display_fibril_entries(
    review_rows,
    ["pdb_id", "species", "deposited_lengths", "pdb_mutations", "pdb_fragment",
     "atlas_origins", "ordered_residues", "publication_url", "decision", "reason"],
     
)

pdb id,species,deposited lengths,pdb mutations,pdb fragment,atlas origins,ordered residues,publication url,decision,reason
10XU,Homo sapiens,67,unknown,unknown,"human, recombinant, produced by seed amplification assay from MSA patient cerebrospinal fluid",35-98,https://doi.org/10.1038/s41467-025-66146-4,include,human entry with a linked manuscript; eligible for metadata curation
12AP,Homo sapiens,140,unknown,unknown,"human, A53T mutation, extracted from Parkinson's Disease patient brain, singlet",31-100,https://doi.org/10.64898/2026.04.07.716933,include,human entry with a linked manuscript; eligible for metadata curation
12AQ,Homo sapiens,140,unknown,unknown,"human, A53T mutation, extracted from Parkinson's Disease patient brain, doublet",31-100,https://doi.org/10.64898/2026.04.07.716933,include,human entry with a linked manuscript; eligible for metadata curation
12AR,Homo sapiens,140,unknown,unknown,"human, A53T mutation, extracted from M83(+/+) mouse, singlet",36-99,https://doi.org/10.64898/2026.04.07.716933,include,human entry with a linked manuscript; eligible for metadata curation
12AT,Homo sapiens,140,unknown,unknown,"human, A53T mutation, extracted from M83(+/+) mouse, doublet",36-99,https://doi.org/10.64898/2026.04.07.716933,include,human entry with a linked manuscript; eligible for metadata curation
32DS,Homo sapiens,141,unknown,unknown,"human, recombinant, pH 7.0, sample 23, type 2A","14-25, 35-97",https://doi.org/10.7554/elife.112692.1,include,human entry with a linked manuscript; eligible for metadata curation
32ED,Homo sapiens,141,unknown,unknown,"human, recombinant, pH 7.0, sample 23, type 2B","14-25,36-96",https://doi.org/10.7554/elife.112692.1,include,human entry with a linked manuscript; eligible for metadata curation
32EP,Homo sapiens,141,unknown,unknown,"human, recombinant, pH 7.0, sample 23, type 5A",1-97,https://doi.org/10.7554/elife.112692.1,include,human entry with a linked manuscript; eligible for metadata curation
32ER,Homo sapiens,98,unknown,unknown,"human, recombinant, pH 7.0, sample 23, type 5B",1-97,https://doi.org/10.7554/elife.112692.1,include,human entry with a linked manuscript; eligible for metadata curation
32HY,Homo sapiens,141,unknown,unknown,"human, recombinant, pH 5.8, type 3C",36-97,https://doi.org/10.7554/elife.112692.1,include,human entry with a linked manuscript; eligible for metadata curation
